# DEBUG: MTS CV Pipeline Test (with multiprocessing)
Runs 1 no-physics combo through the exact same `mp.Pool` + worker path as the real grid search.
5 epochs, h=64, seq_1H=168 — should finish in a few minutes.

In [ ]:
import sys, os
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

current_path = os.getcwd()
library_path = current_path.split('UCB-USACE-RR-PROJECT')[0] + 'UCB-USACE-RR-PROJECT'
sys.path.insert(0, library_path)

from UCB_training.UCB_train import UCB_trainer
from UCB_training.UCB_utils import data_dir, get_yaml_path, fractional_multi_lr
from UCB_training.grid_search_workers import run_single_experiment_nophysics

path_to_csv = data_dir()
path_to_yaml = get_yaml_path('calpella_mtslstm2')
print(f'Data: {path_to_csv}')
print(f'YAML: {path_to_yaml}')

In [ ]:
import multiprocessing as mp
import itertools
from tqdm import tqdm
from datetime import datetime

# --- Minimal grid: 1 combo, 5 epochs ---
hyperparam_space = {
    'hidden_size': [64],
    'output_dropout': [0.4],
    'seq_length_1D': [90],
    'seq_length_1H': [168],
    'num_layers': [1],
    'epochs': [5],
    'batch_size': [64],
    'schedule_pairs': [((0.5, 0.25), (0.01, 0.005, 0.001))],
}

hyperparam_names = list(hyperparam_space.keys())
all_combinations = list(itertools.product(*[hyperparam_space[hp] for hp in hyperparam_names]))
print(f'Combos: {len(all_combinations)}')

BASIN = 'calpella'
RUN_LABEL = 'DEBUG_CV'
RUN_STAMP = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
GPU_SETTING = -1
RUNS_PARENT = os.path.join(os.getcwd(), 'debug_mp_run')
verbose = True

use_cv = True
CV_INTERVAL_MONTH = 'October'
CV_INTERVAL_LENGTH = 2
CV_VALIDATION_LENGTH = 1

print(f'RUN_STAMP: {RUN_STAMP}')
print(f'RUNS_PARENT: {RUNS_PARENT}')

In [ ]:
# --- Run through mp.Pool (same as real grid search) ---
task_args_no = [
    (idx, comb, hyperparam_names, path_to_csv, path_to_yaml,
     GPU_SETTING, RUNS_PARENT, RUN_LABEL, RUN_STAMP, verbose,
     UCB_trainer, fractional_multi_lr, 1, False, False,
     True, True, use_cv, CV_INTERVAL_MONTH, CV_INTERVAL_LENGTH, CV_VALIDATION_LENGTH,
         None, None, None, None,  # train/val start/end
         None, None, None,  # val_eval_start/end, validation_start_per_frequency
         None, None, None,  # train_ranges, validation_ranges, dataset_name
         0, 1, False)  # fold_id, total_folds, cv_external_queue_mode
    for idx, comb in enumerate(all_combinations)
]

num_cores = max(1, mp.cpu_count() - 1)
print(f'Spawning {num_cores} workers for {len(all_combinations)} combo(s)...\n')

with mp.Pool(processes=num_cores) as pool:
    results = list(tqdm(
        pool.imap(run_single_experiment_nophysics, task_args_no),
        total=len(all_combinations),
        desc='Debug No-Physics',
        unit='it',
        ncols=60,
        ascii=True
    ))

df_results = pd.DataFrame(results)
print('\nResults:')
df_results

In [ ]:
import shutil
from pathlib import Path

debug_dir = RUNS_PARENT
if os.path.exists(debug_dir):
    size_mb = sum(f.stat().st_size for f in Path(debug_dir).rglob('*') if f.is_file()) / 1e6
    print(f'Debug dir: {debug_dir} ({size_mb:.1f} MB)')
    # Uncomment to delete:
    # shutil.rmtree(debug_dir)
    # print('Deleted.')